In [22]:
!pip install torch torchvision torchaudio scikit-learn tqdm

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import os

ROOT = "/content/drive/MyDrive/Hackenza_MUPlovers"

FEATURE_DIR = os.path.join(ROOT, "cache/features")
TRAIN_MANIFEST = os.path.join(ROOT, "data/train_manifest_split.csv")

print("Feature dir exists:", os.path.exists(FEATURE_DIR))
print("Manifest exists:", os.path.exists(TRAIN_MANIFEST))

Feature dir exists: True
Manifest exists: True


In [25]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [26]:
ROOT = "/content/drive/MyDrive/Hackenza_MUPlovers"
print(os.listdir(ROOT))

['data', 'cache']


In [27]:
print(os.listdir(os.path.join(ROOT, "data")))

['chunk_index.csv', 'chunk_vad.csv', 'processed', 'val_ids.csv', 'train_ids.csv', 'test_ids.csv', 'train_ids.gsheet', 'test_manifest_split.csv', 'val_manifest_split.csv', 'train_manifest_split.csv']


In [28]:
for root, dirs, files in os.walk(ROOT):
    for file in files:
        if "manifest" in file.lower():
            print(os.path.join(root, file))

/content/drive/MyDrive/Hackenza_MUPlovers/data/test_manifest_split.csv
/content/drive/MyDrive/Hackenza_MUPlovers/data/val_manifest_split.csv
/content/drive/MyDrive/Hackenza_MUPlovers/data/train_manifest_split.csv


In [29]:
import pandas as pd

df = pd.read_csv(TRAIN_MANIFEST)
print(df.columns)
df.head()

Index(['file_id', 'audio_url', 'nativity_status', 'label', 'language',
       'raw_path', 'processed_path'],
      dtype='object')


,file_id,audio_url,nativity_status,label,language,raw_path,processed_path
0,716,https://d2r5sftrozu18z.cloudfront.net/uploads/...,Non-Native,0,Arabic_QA,data/raw/716.wav,data/processed/716.wav
1,668,https://d2r5sftrozu18z.cloudfront.net/uploads/...,Native,1,Arabic_SA,data/raw/668.wav,data/processed/668.wav
2,408,https://d2r5sftrozu18z.cloudfront.net/uploads/...,Non-Native,0,Arabic_KW,data/raw/408.wav,data/processed/408.wav
3,440,https://d2r5sftrozu18z.cloudfront.net/uploads/...,Native,1,Arabic_SA,data/raw/440.wav,data/processed/440.wav
4,492,https://d2r5sftrozu18z.cloudfront.net/uploads/...,Native,1,Arabic_SA,data/raw/492.wav,data/processed/492.wav


In [30]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset

class NativeDataset(Dataset):
    def __init__(self, manifest_csv, feature_dir):
        self.df = pd.read_csv(manifest_csv)
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_id = str(row["file_id"])
        label = float(row["label"])

        feature_path = os.path.join(self.feature_dir, f"{file_id}.npy")
        x = np.load(feature_path)
        x = torch.tensor(x, dtype=torch.float32)

        return x, torch.tensor(label, dtype=torch.float32)

In [31]:
from torch.nn.utils.rnn import pad_sequence
import torch

def collate_fn(batch):
    sequences, labels = zip(*batch)

    lengths = [seq.shape[0] for seq in sequences]
    padded = pad_sequence(sequences, batch_first=True)

    mask = torch.zeros(padded.shape[:2])
    for i, length in enumerate(lengths):
        mask[i, :length] = 1

    labels = torch.stack(labels)

    return padded, mask, labels

In [32]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    sequences, labels = zip(*batch)

    lengths = [seq.shape[0] for seq in sequences]

    padded = pad_sequence(sequences, batch_first=True)  # [B, T_max, 783]

    mask = torch.zeros(padded.shape[:2])

    for i, length in enumerate(lengths):
        mask[i, :length] = 1

    labels = torch.stack(labels)

    return padded, mask, labels

In [33]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

In [34]:
x, mask, y = next(iter(train_loader))
print(x.shape, mask.shape, y.shape)

torch.Size([1, 30, 783]) torch.Size([1, 30]) torch.Size([1])


In [35]:
from torch.utils.data import DataLoader

train_dataset = NativeDataset(
    os.path.join(ROOT, "data/train_manifest_split.csv"),
    FEATURE_DIR
)

val_dataset = NativeDataset(
    os.path.join(ROOT, "data/val_manifest_split.csv"),
    FEATURE_DIR
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn
)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

Train size: 112
Val size: 24


In [36]:
import torch.nn as nn
import torch.nn.functional as F

class TemporalEncoder(nn.Module):
    def __init__(self, input_dim=783, hidden_dim=256):
        super().__init__()
        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.output_dim = hidden_dim * 2

    def forward(self, x):
        out, _ = self.gru(x)
        return out


class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1)
        scores = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return pooled


class AccentModel(nn.Module):
    def __init__(self, input_dim=783):
        super().__init__()
        self.encoder = TemporalEncoder(input_dim)
        self.pool = AttentionPooling(self.encoder.output_dim)
        self.classifier = nn.Sequential(
            nn.Linear(self.encoder.output_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x, mask):
        seq = self.encoder(x)
        pooled = self.pool(seq, mask)
        logits = self.classifier(pooled).squeeze(-1)
        return logits

In [37]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

model = AccentModel(783).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [38]:
def train_epoch():
    model.train()
    total_loss = 0

    for x, mask, y in tqdm(train_loader):
        x, mask, y = x.to(device), mask.to(device), y.to(device)

        optimizer.zero_grad()

        logits = model(x, mask)
        loss = F.binary_cross_entropy_with_logits(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


def evaluate():
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, mask, y in val_loader:
            x, mask = x.to(device), mask.to(device)

            logits = model(x, mask)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds)

    return acc, f1

In [39]:
for epoch in range(10):
    loss = train_epoch()
    acc, f1 = evaluate()

    print(f"Epoch {epoch+1}")
    print("Loss:", loss)
    print("Val Accuracy:", acc)
    print("Val F1:", f1)
    print("-"*40)

100%|██████████| 14/14 [00:02<00:00,  6.94it/s]


Epoch 1
Loss: 0.6390933671167919
Val Accuracy: 0.7083333333333334
Val F1: 0.8292682926829268
----------------------------------------


100%|██████████| 14/14 [00:01<00:00,  9.41it/s]


Epoch 2
Loss: 0.5976034211260932
Val Accuracy: 0.7083333333333334
Val F1: 0.8292682926829268
----------------------------------------


100%|██████████| 14/14 [00:01<00:00, 10.10it/s]


...

Epoch 8
Loss: 0.5952279695442745
Val Accuracy: 0.7083333333333334
Val F1: 0.8292682926829268
----------------------------------------


100%|██████████| 14/14 [00:01<00:00, 10.67it/s]


Epoch 9
Loss: 0.581061356834003
Val Accuracy: 0.7083333333333334
Val F1: 0.8292682926829268
----------------------------------------


100%|██████████| 14/14 [00:00<00:00, 19.02it/s]


Epoch 10
Loss: 0.5566821502787727
Val Accuracy: 0.6666666666666666
Val F1: 0.8
----------------------------------------
